# Natural Language Processing — Assignment 3
## Building a Model Context Protocol (MCP) Server for an AI Assistant
**Student Name:** Pramodh  
**Student ID:** B210195  
**Section / Class:** C4  
**Date:** November 24, 2026  

---
### 1. Abstract & Protocol Overview
The **Model Context Protocol (MCP)**, open-sourced by Anthropic, provides an open standard for safely connecting foundation AI models with external tools, contextual resources, and prompt templates.
Instead of relying on brittle proprietary plugins, MCP establishes a bidirectional client-server architecture communicating via **JSON-RPC 2.0** over standard input/output (`stdio`) or Server-Sent Events (`SSE`).

In this assignment, we build and verify a production-grade MCP Server exposing:
- **Tools:** `query_database`, `calculate_financial_metrics`, `search_assistant_docs`, `system_health_check`.
- **Resources:** `assistant://system_config`, `assistant://database_schema`, `assistant://audit_logs`.
- **Prompts:** `financial_analysis_prompt`, `system_diagnostic_prompt`.



In [1]:
# 1. Imports and Verification
import json
import subprocess
import sys
from mcp_client_test import MCPTestClient

print("MCP Client Environment Loaded Successfully!")



### 2. Protocol Handshake (`initialize`)
We spawn the MCP server as a subprocess and negotiate the capabilities handshake.



In [2]:
client = MCPTestClient()
init_res = client.send_request("initialize", {
    "protocolVersion": "2024-11-05",
    "capabilities": {},
    "clientInfo": {"name": "NLP-Notebook-Client", "version": "1.0.0"}
})
print("Handshake Response:")
print(json.dumps(init_res, indent=2))
client.send_notification("notifications/initialized")



### 3. Tool Discovery and Execution
Querying available tools and invoking `query_database` and `calculate_financial_metrics`.



In [3]:
# List Tools
tools = client.send_request("tools/list")["result"]["tools"]
print(f"Discovered Tools: {[t['name'] for t in tools]}")

# Invoke SQL Tool
sql_res = client.send_request("tools/call", {
    "name": "query_database",
    "arguments": {"query": "SELECT department, revenue, expenses FROM metrics LIMIT 3"}
})
print("\nDatabase Query Output:")
print(sql_res["result"]["content"][0]["text"])

# Invoke Math Tool
cagr_res = client.send_request("tools/call", {
    "name": "calculate_financial_metrics",
    "arguments": {"metric_type": "cagr", "initial_value": 1450000.0, "final_value": 1820000.0, "years": 1.0}
})
print("\nFinancial Calculation Output:")
print(cagr_res["result"]["content"][0]["text"])



### 4. Reading Contextual Resources and Prompt Templates
Inspecting passive server resources (`assistant://system_config`) and templated prompts.



In [4]:
# Read Resource
res = client.send_request("resources/read", {"uri": "assistant://system_config"})
print("System Config Resource:")
print(res["result"]["contents"][0]["text"])

# Get Prompt Template
prompt = client.send_request("prompts/get", {
    "name": "financial_analysis_prompt",
    "arguments": {"department": "Cloud AI"}
})
print("\nRendered Prompt:")
print(json.dumps(prompt["result"], indent=2))

client.close()
print("\nMCP Server verification successfully concluded.")



### 5. Architectural Compliance & Security Guardrails
- **Protocol Conformance:** Strict adherence to MCP specification `v2024-11-05` and JSON-RPC 2.0.
- **Security Sandboxing:** Read-only SQL policy enforced, prohibiting `DROP`, `DELETE`, or `UPDATE`.
- **Audit Logging:** Every tool call is automatically stamped and committed into the relational audit log.

